# Problem 43: Multi-Agent Shared Memory

**Difficulty:** Hard | **Framework:** LangGraph

**Categories:** memory, multi-agent

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/multi-agent-shared-memory).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- All agents must be able to read from the shared scratchpad
- Writes to the shared scratchpad must not overwrite each other silently
- Each entry must track which agent wrote it
- The shared memory must be readable by all agents at any point in the workflow


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# BUG: Each agent has its own isolated memory — no shared scratchpad
class State(TypedDict):
    task: str
    result: str

def researcher(state: State) -> State:
    # Researcher finds data but can't share it with others
    messages = [
        SystemMessage(content="You are a researcher. Find key data points."),
        HumanMessage(content=state["task"]),
    ]
    response = llm.invoke(messages)
    return {"result": response.content}

def analyst(state: State) -> State:
    # Analyst has no access to researcher's findings
    messages = [
        SystemMessage(content="You are an analyst. Analyze the data."),
        HumanMessage(content=f"Analyze: {state['result']}"),
    ]
    response = llm.invoke(messages)
    return {"result": response.content}

def writer(state: State) -> State:
    # Writer only sees analyst output, not researcher's raw data
    messages = [
        SystemMessage(content="You are a technical writer. Write a report."),
        HumanMessage(content=f"Write report from: {state['result']}"),
    ]
    response = llm.invoke(messages)
    return {"result": response.content}

graph = StateGraph(State)
graph.add_node("researcher", researcher)
graph.add_node("analyst", analyst)
graph.add_node("writer", writer)
graph.add_edge(START, "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", END)
app = graph.compile()

result = app.invoke({"task": "Research renewable energy trends and write a report."})
print(result["result"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Use a shared list or dictionary that all agents can append to, with each entry tagged by the writing agent's name
2. Instead of overwriting, use an append-only log pattern where each agent adds entries with metadata
3. Inject the current state of the shared scratchpad into each agent's context before it runs


## 7. Evaluation Criteria

Check your solution against these criteria:

- All agents can read shared memory
- Writes are attributed
- No silent overwrites
- Downstream agents see upstream data
